# 97 — Campaign B post-M2 pipeline

分割レーン設計の Lane B–D（screening → M6、単一 GPU）。

- M2 builder レーンは将来；`M2_READY.json` は検出のみ
- `WAITING_FOR_M2` reconciler は TODO（driver 未配線）
- **96 と同時起動しない**（GPU 競合）

設計: `docs/campaign_b_parallel_split_design.md`


In [ ]:
from pathlib import Path
import os, sys, json

BUNDLE_NAME = 'validated_4d_su2_rg_codex_bundle'
explicit = os.environ.get('VALIDATED_RG_PROJECT_ROOT')
candidates = ([Path(explicit)] if explicit else []) + [
    Path.cwd(), Path.cwd().parent, Path.cwd() / BUNDLE_NAME,
    Path('/notebooks') / BUNDLE_NAME, Path('/notebooks'),
]
PROJECT_ROOT = next((
    p.expanduser().resolve()
    for p in candidates
    if (p.expanduser() / 'src' / 'campaign_b' / 'post_m2_pipeline.py').is_file()
), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Pull latest main; src/campaign_b/post_m2_pipeline.py not found.')
PERSIST_ROOT = Path(
    os.environ.get('VALIDATED_RG_PERSIST_ROOT', '/storage/validated_4d_su2_rg')
).expanduser().resolve()
os.environ['VALIDATED_RG_PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ['VALIDATED_RG_PERSIST_ROOT'] = str(PERSIST_ROOT)
os.environ['VALIDATED_RG_DISABLE_SESSION_WALLCLOCK'] = '1'
os.environ.setdefault('VALIDATED_RG_PERSIST_ACK', 'I_CONFIRM_THIS_PATH_IS_PERSISTENT')
os.environ.setdefault('VALIDATED_RG_M3_ALLOW_CODE_DRIFT', '1')
os.environ.setdefault('VALIDATED_RG_M4_ALLOW_CODE_DRIFT', '1')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
print('PROJECT_ROOT', PROJECT_ROOT)
print('PERSIST_ROOT', PERSIST_ROOT)
if not torch.cuda.is_available():
    raise RuntimeError('Notebook 97 requires CUDA (single GPU lane).')
print('GPU', torch.cuda.get_device_properties(0).name)


In [ ]:
from src.runtime import validate_persistent_root
from src.campaign_b.post_m2_pipeline import find_m2_ready_markers

validate_persistent_root()
print('M2_READY markers:', json.dumps(find_m2_ready_markers(PERSIST_ROOT), indent=2, default=str))
SELECTED_BACKLOG_TARGET = 8
SCREENING_CHUNK_SIZE = 32
MAX_ROUNDS = 50


In [ ]:
from src.campaign_b.post_m2_pipeline import run_post_m2_pipeline

SESSION = run_post_m2_pipeline(
    persistent_root=PERSIST_ROOT,
    project_root=PROJECT_ROOT,
    selected_backlog_target=SELECTED_BACKLOG_TARGET,
    screening_chunk_size=SCREENING_CHUNK_SIZE,
    max_rounds=MAX_ROUNDS,
)
print(json.dumps({
    'session_id': SESSION.get('session_id'),
    'm2_ready_count': SESSION.get('m2_ready_count'),
    'waiting_for_m2_todo': SESSION.get('waiting_for_m2_todo'),
    'end_to_end': SESSION.get('end_to_end'),
    'certification_status': SESSION.get('certification_status'),
    'claim_scope': SESSION.get('claim_scope'),
}, indent=2, ensure_ascii=False, default=str))
